[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/slip_systems/slip_systems.ipynb)

# Slip in Metal Crystals

This example computes which slip systems of a metal crystal yield under tension, and how slipping rotates the crystal relative to the load. When a copper wire bends permanently, layers of atoms inside each grain slide over one another along specific crystal planes and directions that form its slip systems. Tension resolves a different shear stress onto each slip system, so the most strongly sheared system yields first. As that system slips, the tensile axis turns relative to the lattice until a second system reaches the same resolved shear stress and the two slip together.

A slip plane through the origin is modelled as an extrinsic vector, while atom offsets, slip directions, and the tensile axis are modelled as intrinsic antivectors. Joining two nearest-neighbour edge directions `directions[:, 0] & directions[:, 1]` and multiplying by the tetrahedron's intrinsic winding gives each oriented slip plane. Tension along `axis` places a traction covector `axis.dual_inverse() * (planes & axis)` on each plane, and pairing it with a slip direction `tractions & directions` gives the resolved shear stress per unit tension, which is called the Schmid factor. Each slip shears crystal directions by `Direction + amount * direction * (plane & Direction)`, and the resulting displacement `move` rotates the tensile axis in the plane spanned by `axis.commutator(move)`.

In Miller-index and Cartesian tensor notation, face-centred cubic slip acts on $\{111\}$ planes with unit normals $n_i^\alpha$ along $\langle 111 \rangle$ and unit Burgers directions $s_i^\alpha$ along $\langle 110 \rangle$, uniaxial tension $\sigma_{ij} = \sigma\, t_i t_j$ gives the Schmid factor $m^\alpha = t_i s_i^\alpha\, n_j^\alpha t_j = \cos\lambda^\alpha \cos\phi^\alpha$, and the plastic velocity gradient $L_{ij}^p = \sum_\alpha \dot\gamma^\alpha s_i^\alpha n_j^\alpha$ splits into symmetric stretching $D_{ij}^p = \tfrac{1}{2}(L_{ij}^p + L_{ji}^p)$ and skew plastic spin $W_{ij}^p = \tfrac{1}{2}(L_{ij}^p - L_{ji}^p)$ to evolve the tensile axis via $\dot t_i = L_{ij}^p t_j - (t_k D_{kl}^p t_l)\, t_i$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.mechanics.slip_systems import render
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
Plane = ga.gatype.vector()                                 # Vector+
Direction = ga.gatype.antivector()                         # Antivector-

## Twelve slip systems

Sliding one half of a crystal across a plane preserves the lattice whenever the step carries every atom onto another lattice site. The shortest lattice steps connect nearest-neighbour atoms and form the slip directions, while the densest atomic planes hold three of those directions each and form the slip planes. In a face-centred cubic crystal, four mutually neighbouring atoms form a regular tetrahedron whose six edge differences give all six slip directions and whose four faces give all four slip planes. Joining two slip directions in a face `directions[:, 0] & directions[:, 1]` and multiplying by the tetrahedron's intrinsic winding orients each slip plane consistently, producing twelve slip systems in total.

In [ ]:
# Four mutually neighbouring lattice sites of a face-centred cubic crystal, each opposite its slip plane.
atom_positions = [[0, 0, 0], [0, 1, 1], [1, 1, 0], [1, 0, 1]]

In [ ]:
def slip_directions(atoms: Direction) -> Direction:
    """Nearest-neighbour atom differences along shared tetrahedral edges give the three unit slip directions (`[4, 3] Direction-`) in each slip plane."""
    others = (np.arange(4)[:, None] + np.arange(1, 4)) % 4                    # [4, 3]
    # Each plane shares a slip direction along the edge joining the two atoms that neither plane lies opposite.
    ends = np.array([[sorted(set(range(4)) - {plane, other}) for other in row]
                     for plane, row in enumerate(others)])                    # [4, 3, 2]
    return (atoms[ends[..., 1]] - atoms[ends[..., 0]]).normalized()           # [4, 3] Direction-


def slip_planes(atoms: Direction, directions: Direction) -> Plane:
    """Joining two slip directions in each tetrahedral face and multiplying by the atom winding orients the four unit slip planes (`[4] Plane+`)."""
    winding = (atoms[1] - atoms[0]) & (atoms[2] - atoms[0]) & (atoms[3] - atoms[0])   # [] Scalar-
    return ((directions[:, 0] & directions[:, 1]) * winding).normalized()             # [4] Plane+


atoms = mv.vector(atom_positions).dual()                                      # [4] Direction-
slips = slip_directions(atoms)                                                # [4, 3] Direction-
planes = slip_planes(atoms, slips)                                            # [4] Plane+

## The Schmid factor

Unit tension along `axis` exerts a traction covector `axis.dual_inverse() * (planes & axis)` across each slip plane, using the Euclidean metric once through `axis.dual_inverse()` to convert the tensile direction into its work-conjugate covector. Pairing that traction with each slip direction `tractions & directions` gives the Schmid factor, which reaches one half when the tensile axis makes equal forty-five-degree angles with both the plane normal and the slip direction. Stereographic reflection `(z + offsets).normalized() >> z.dual()` lifts a square grid on the equatorial plane to pull directions across the hemisphere, and along `[001]` eight of the twelve slip systems tie at one over the square root of six.

In [ ]:
n_samples = 1000

In [ ]:
def schmid_factors(axis: Direction, planes: Plane, directions: Direction) -> Scalar:
    """Pairing the plane tractions of unit tension along `axis` with each slip direction gives the Schmid factors (`[..., 4, 3] Scalar+`)."""
    # Unit tension along each axis places a work-conjugate traction covector on each slip plane.
    tractions = axis.dual_inverse()[..., None] * (planes & axis[..., None])   # [..., 4] Plane+
    return tractions[..., None] & directions                                  # [..., 4, 3] Scalar+


def disk_directions(n_samples: int) -> Direction:
    """Stereographic reflection across `[001]` lifts a square grid on the equatorial plane to pull directions (`[n_samples, n_samples] Direction-`)."""
    across = np.linspace(-1, 1, n_samples)                                    # [n_samples]
    offsets = mv.x * across + mv.y * across[:, None]                          # [n_samples, n_samples] Plane+
    mirrors = (mv.z + offsets).normalized()                                   # [n_samples, n_samples] Plane+
    return mirrors >> mv.z.dual()                                             # [n_samples, n_samples] Direction-


directions = disk_directions(n_samples)                                       # [n_samples, n_samples] Direction-
factors = schmid_factors(directions, planes, slips)                           # [n_samples, n_samples, 4, 3] Scalar+
cube_factors = schmid_factors(mv.z.dual(), planes, slips).abs()               # [4, 3] Scalar+
print("Schmid factors along [001]:\n", cube_factors.to_array().round(4))

## Lattice rotation under tension

Slipping on one system displaces each crystal direction along `direction` in proportion to its height `plane & axis` above the slip plane, and simultaneous slips on several systems sum their displacements. Because a metal's slip rate rises as a high power of resolved shear stress, the leading system carries nearly all the slip until a second system matches its Schmid factor, whereupon both share each step equally. Gripping the specimen keeps the tensile axis fixed in the laboratory, so in the crystal frame the tensile axis rotates toward the active slip direction through `(axis.commutator(move) / 2).exp() >> axis`, while radial stretching along `axis` drops out of the commutator.

In [ ]:
# Initial tensile axes inside the standard triangle between [001], [011], and [111].
start_directions = [
    [0.05, 0.3, 1], [0.15, 0.32, 1], [0.03, 0.5, 1], [0.2, 0.5, 1],
    [0.05, 0.75, 1], [0.3, 0.6, 1], [0.12, 0.62, 1], [0.4, 0.55, 1],
]
n_starts = len(start_directions)
step_slip = 0.01
steepness = 30
n_steps = 500
n_frames = 88
frame_duration_ms = 40

In [ ]:
def slip_amounts(factors: Scalar, step_slip: float, steepness: int) -> Scalar:
    """A power-law flow rule scales each system's slip increment (`[..., 4, 3] Scalar+`) by its Schmid factor relative to the leading system."""
    magnitudes = factors.abs()                                                # [..., 4, 3] Scalar+
    leading = magnitudes.max(axis=(-2, -1), keepdims=True)                    # [..., 1, 1] Scalar+
    return step_slip * factors * (magnitudes ** (steepness - 1) / leading ** steepness)   # [..., 4, 3] Scalar+


def turned_axes(axis: Direction, planes: Plane, directions: Direction, amounts: Scalar) -> Direction:
    """Simultaneous slips `amounts` shear `axis` and rotate each tensile axis (`[...] Direction-`) in the crystal frame."""
    # Each slip direction enters both `schmid_factors` and `moves`, so arbitrary sign choices on edge directions cancel out.
    moves = amounts * directions * (planes[:, None] & axis[..., None, None])  # [..., 4, 3] Direction-
    move = moves.sum(axis=-1).sum(axis=-1)                                    # [...] Direction-
    # The commutator extracts the rotation plane spanned by `axis` and `move`, discarding radial stretch along `axis`.
    rotors = (axis.commutator(move) / 2).exp()                                # [...] Rotor+
    return rotors >> axis                                                     # [...] Direction-

In [ ]:
def slip_steps(
    axis: Direction,
    planes: Plane,
    directions: Direction,
    step_slip: float,
    steepness: int,
    n_steps: int,
) -> Iterator[tuple[Direction, Scalar]]:
    """Yields the rotated tensile axes (`[...] Direction-`) and system slip increments (`[..., 4, 3] Scalar+`) over `n_steps` steps."""
    for _ in range(n_steps):
        amounts = slip_amounts(schmid_factors(axis, planes, directions), step_slip, steepness)   # [..., 4, 3] Scalar+
        axis = turned_axes(axis, planes, directions, amounts)                                    # [...] Direction-
        yield axis, amounts


start_axes = mv.vector(start_directions).normalized().dual()                  # [n_starts] Direction-
schmid = partial(schmid_factors, planes=planes, directions=slips)
states = show_states(
    slip_steps(start_axes, planes, slips, step_slip, steepness, n_steps),
    render.animate_paths, schmid, start_axes, n_steps, n_frames,
    name="slip_systems_paths", duration_ms=frame_duration_ms, n_frames=n_frames,
)

In [ ]:
stepped_axes, step_amounts = zip(*states)
axes = stack([start_axes, *stepped_axes])                                     # [n_steps + 1, n_starts] Direction-
amounts = stack(step_amounts)                                                 # [n_steps, n_starts, 4, 3] Scalar+
system_slip = amounts.abs().sum(axis=0).reshape(n_starts, 12)                 # [n_starts, 12] Scalar+
order = system_slip.argsort(axis=-1)                                          # [n_starts, 12]
two_system_share = (
    system_slip[np.arange(n_starts)[:, None], order[:, -2:]].sum(axis=-1) / system_slip.sum(axis=-1)
)                                                                             # [n_starts] Scalar+
print("Share of each pull's slip on its two leading systems:", two_system_share.to_array().round(3))
save_figure(render.draw_chart(schmid, planes, slips, axes, amounts), "slip_systems");

On the full stereographic disk at left, seen down `[001]`, each colour marks the slip system with the largest Schmid factor, fading toward white around the dark squares of the four plane normals and deepening between them and the dots of the six slip directions. Cube symmetry tiles the disk with copies of the standard triangle between `[001]`, `[011]`, and `[111]`, shown enlarged at right and in the animation above. Every trajectory inside the triangle starts in the orange single-slip region and turns along a great circle toward its slip direction `[101]` until meeting the boundary between `[001]` and `[111]`, where the brown conjugate system matches its Schmid factor and the two systems drive the tensile axis together along the boundary to `[112]`.

In [ ]:
# checks
goal = mv.vector([1, 1, 2]).normalized().dual()                               # Direction-
assert ((planes[:, None] & slips).abs() < 1e-12).all()
assert (factors.abs() <= 0.5 + 1e-12).all()
assert (cube_factors.max() - 1 / np.sqrt(6)).abs() < 1e-12
assert ((cube_factors - cube_factors.max()).abs() < 1e-12).sum() == 8
assert ((axes[-1] - goal).norm() < 0.02).all()
assert (two_system_share > 0.95).all()